<a href="https://colab.research.google.com/github/Coding-7allhh/YT_Auto/blob/main/Part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ==============================================================
# 🚀 SUPER POWER - LONG VIDEO -> MULTIPLE 1:20 CLIPS
# Google Colab (Local Mode with gdown)
#
# INPUT:
#   /content/download/<TYPE>/*.mp4 (Downloaded via gdown)
#
# OUTPUT:
#   /content/clips/<TYPE>/
#
# FEATURES:
# ✅ 10 videos parallel processing
# ✅ 80-second (1:20) clips
# ✅ Full HD resolution preserved
# ✅ Audio preserved
# ✅ 30 FPS -> 60 FPS
# ✅ 60 FPS source -> no unnecessary FPS conversion
# ✅ NVIDIA GPU auto-detection
# ✅ GPU encoding when NVENC is available
# ✅ CPU optimized fallback
# ✅ Existing complete videos skipped
# ✅ Incomplete final clip removed
# ✅ Multiple TYPE folders automatically detected
# ✅ Original long videos are NEVER deleted
# ==============================================================


import gdown
from pathlib import Path

# Install gdown if needed
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "gdown"],
    check=True
)

# Google Drive folder IDs
FOLDER_1 = "1l9ayaXiRBwUr1Hrjag7r3CycVDiN24fh"
FOLDER_2 = "1VPSEN4sticaR73-RNhZLJXOm7RBg0vKS"

# Download locations

Path("download").mkdir(parents=True, exist_ok=True)
Path("clips").mkdir(parents=True, exist_ok=True)

# Download both folders
gdown.download_folder(
    id=FOLDER_1,
    output="assets",
    quiet=False
)

gdown.download_folder(
    id=FOLDER_2,
    output="download",
    quiet=False
)

print("Both folders downloaded successfully!")





import subprocess
import sys

print("Checking system dependencies...")

# FFmpeg is installed by GitHub Actions using sudo.
ffmpeg_check = subprocess.run(
    ["ffmpeg", "-version"],
    capture_output=True,
    text=True
)

if ffmpeg_check.returncode != 0:
    raise RuntimeError(
        "FFmpeg is missing. Install it in GitHub Actions first."
    )

print("FFmpeg:", ffmpeg_check.stdout.splitlines()[0])

# Install Python packages without administrator permissions.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "gdown"],
    check=True
)

print("All dependencies are ready!")



import subprocess
import sys

# ==============================================================
# 1) INSTALL FFMPEG & GDOWN
# ==============================================================

print("🔄 Installing system dependencies...")
subprocess.run("apt-get -qq update", shell=True, check=True)
subprocess.run("apt-get -qq install -y ffmpeg", shell=True, check=True)
subprocess.run("pip install -q --upgrade gdown", shell=True, check=True)


# ==============================================================
# 2) IMPORTS
# ==============================================================

import os
import re
import math
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import gdown


# ==============================================================
# 3) DIRECTORY CONFIGURATION (LOCAL PATHS)
# ==============================================================

INPUT_ROOT = Path("/content/download")
OUTPUT_ROOT = Path("/content/clips")


# ==============================================================
# 4) DOWNLOAD INPUT FOLDER FROM PUBLIC GOOGLE DRIVE LINK VIA GDOWN
# ==============================================================

FOLDER_ID = "1l9ayaXiRBwUr1Hrjag7r3CycVDiN24fh-"

try:
    INPUT_ROOT.mkdir(parents=True, exist_ok=True)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    print("\n🔄 Downloading shared folder contents from Google Drive using gdown...")
    # Using gdown Python API correctly instead of placing it incorrectly in subprocess.run
    gdown.download_folder(
        id=FOLDER_ID,
        output=str(INPUT_ROOT),
        quiet=False,
        use_cookies=False
    )
    print("✅ Download completed successfully.")
except Exception as e:
    print(f"⚠️ Error during folder download: {e}")


# ==============================================================
# 5) MAIN SETTINGS
# ==============================================================

CLIP_LENGTH = 80
MAX_PARALLEL_VIDEOS = 10
TARGET_FPS = 60
SMOOTH_INTERPOLATION = False


# ==============================================================
# 6) SUPPORTED VIDEO TYPES
# ==============================================================

VIDEO_EXTENSIONS = {
    ".mp4",
    ".mkv",
    ".mov",
    ".webm",
    ".avi",
    ".m4v"
}


# ==============================================================
# 8) CHECK NVIDIA GPU / NVENC
# ==============================================================

def check_nvenc():
    try:
        gpu = subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        if gpu.returncode != 0:
            return False
        encoders = subprocess.run(
            [
                "ffmpeg",
                "-hide_banner",
                "-encoders"
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )
        return "h264_nvenc" in encoders.stdout
    except Exception:
        return False

USE_NVENC = check_nvenc()

print("\n" + "=" * 72)
print("🚀 VIDEO CLIP ENGINE")
print("=" * 72)

if USE_NVENC:
    print("✅ NVIDIA NVENC GPU detected")
    print("⚡ GPU encoding will be used")
else:
    print("ℹ️ NVENC not available")
    print("⚙️ CPU optimized mode will be used")


# ==============================================================
# 9) GET VIDEO INFORMATION
# ==============================================================

def get_video_info(video_path):
    cmd = [
        "ffprobe",
        "-v",
        "error",
        "-select_streams",
        "v:0",
        "-show_entries",
        "stream=width,height,r_frame_rate,avg_frame_rate",
        "-show_entries",
        "format=duration",
        "-of",
        "default=noprint_wrappers=1",
        str(video_path)
    ]
    try:
        result = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        if result.returncode != 0:
            return None
        data = {}
        for line in result.stdout.splitlines():
            if "=" in line:
                key, value = line.split("=", 1)
                data[key] = value.strip()
        duration = float(data.get("duration", "0"))
        fps_string = data.get("avg_frame_rate", "30/1")
        if "/" in fps_string:
            num, den = fps_string.split("/")
            num = float(num)
            den = float(den)
            fps = num / den if den != 0 else 30.0
        else:
            fps = float(fps_string)
        width = int(data.get("width", "0"))
        height = int(data.get("height", "0"))
        return {
            "duration": duration,
            "fps": fps,
            "width": width,
            "height": height
        }
    except Exception:
        return None


# ==============================================================
# 10) GET EXPECTED CLIP COUNT
# ==============================================================

def get_full_clip_count(duration):
    if duration < CLIP_LENGTH:
        return 0
    return int(duration // CLIP_LENGTH)


# ==============================================================
# 11) CHECK EXISTING CLIPS
# ==============================================================

def all_clips_exist(output_folder, video_stem, clip_count):
    if clip_count <= 0:
        return False
    for i in range(1, clip_count + 1):
        clip_file = (
            output_folder /
            f"{video_stem}_clip_{i:03d}.mp4"
        )
        if not clip_file.exists():
            return False
        if clip_file.stat().st_size < 100 * 1024:
            return False
    return True


# ==============================================================
# 12) DELETE OLD CLIPS FOR THIS VIDEO
# ==============================================================

def delete_old_clips(output_folder, video_stem):
    for file in output_folder.glob(f"*clip_*.mp4"):
        if video_stem in file.name:
            try:
                file.unlink()
            except Exception:
                pass


# ==============================================================
# 13) BUILD VIDEO FILTER
# ==============================================================

def build_video_filter(source_fps):
    if source_fps >= 59.5:
         return None
    if not SMOOTH_INTERPOLATION:
        return f"fps={TARGET_FPS}"
    return (
        "minterpolate="
        "fps=60:"
        "mi_mode=mci:"
        "mc_mode=aobmc:"
        "me_mode=bidir:"
        "vsbmc=1"
    )


# ==============================================================
# 14) BUILD FFMPEG COMMAND
# ==============================================================

def build_ffmpeg_command(
    input_file,
    output_pattern,
    source_fps
):
    filter_string = build_video_filter(source_fps)
    command = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel", "error",
        "-i",
        str(input_file),
        "-map",
        "0:v:0",
        "-map",
        "0:a?",
        "-map_metadata",
        "0",
        "-force_key_frames",
        f"expr:gte(t,n_forced*{CLIP_LENGTH})",
        "-f",
        "segment",
        "-segment_time",
        str(CLIP_LENGTH),
        "-segment_start_number",
        "1",
        "-reset_timestamps",
        "1",
        "-segment_format",
        "mp4",
        "-movflags",
        "+faststart"
    ]

    if USE_NVENC:
        command += [
            "-c:v",
            "h264_nvenc",
            "-preset",
            "p4",
            "-rc",
            "vbr",
            "-cq",
            "18",
            "-b:v",
            "0",
            "-pix_fmt",
            "yuv420p"
        ]
    else:
        command += [
            "-c:v",
            "libx264",
            "-preset",
            "veryfast",
            "-crf",
            "18",
            "-pix_fmt",
            "yuv420p",
            "-threads",
            "2"
        ]

    if filter_string:
        command += [
            "-vf",
            filter_string,
            "-r",
            str(TARGET_FPS)
        ]

    command += [
        "-c:a",
        "aac",
        "-b:a",
        "128k",
        "-ar",
        "48000",
        str(output_pattern)
    ]
    return command


# ==============================================================
# 15) REMOVE INCOMPLETE FINAL CLIP
# ==============================================================

def cleanup_extra_clips(
    output_folder,
    video_stem,
    expected_count
):
    existing = sorted(output_folder.glob("*.mp4"))
    for file in existing:
        if video_stem in file.name and "_clip_" in file.name:
            match = re.search(r"_clip_(\d+)\.mp4$", file.name, re.IGNORECASE)
            if match:
                number = int(match.group(1))
                if number > expected_count:
                    try:
                        file.unlink()
                        print(f"🗑️ Removed incomplete clip: {file.name}")
                    except Exception:
                        pass


# ==============================================================
# 16) PROCESS ONE VIDEO
# ==============================================================

def process_video(video_file):
    type_name = video_file.parent.name
    output_folder = OUTPUT_ROOT / type_name

    try:
        output_folder.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        return {
            "file": video_file.name,
            "status": f"ERROR: cannot create output directory {e}"
        }

    video_stem = video_file.stem

    print("\n" + "-" * 72)
    print(f"🎬 {video_file.name}")
    print(f"📁 Type: {type_name}")

    info = get_video_info(video_file)
    if info is None:
        return {
            "file": video_file.name,
            "status": "ERROR: cannot read video"
        }

    duration = info["duration"]
    fps = info["fps"]
    width = info["width"]
    height = info["height"]
    expected_clips = get_full_clip_count(duration)

    print(f"📐 Resolution: {width}x{height}")
    print(f"🎞️ FPS: {fps:.2f}")
    print(f"⏱️ Duration: {duration:.2f}s")
    print(f"✂️ Full clips: {expected_clips}")

    if expected_clips == 0:
        return {
            "file": video_file.name,
            "status": "SKIPPED: shorter than 1:20"
        }

    if all_clips_exist(output_folder, video_stem, expected_clips):
        print("⏭️ All clips already exist")
        return {
            "file": video_file.name,
            "status": "SKIPPED: already processed",
            "clips": expected_clips
        }

    delete_old_clips(output_folder, video_stem)

    # We render directly to local storage first
    temp_root = Path("/content/video_clip_temp")
    temp_folder = temp_root / type_name / video_stem
    try:
        temp_folder.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        return {
            "file": video_file.name,
            "status": f"ERROR: temp dir creation failed: {e}"
        }

    # Use generic segment naming, we rename them to match video_stem afterwards
    output_pattern = temp_folder / "segment_%03d.mp4"
    command = build_ffmpeg_command(video_file, output_pattern, fps)

    print("⚡ Processing segments...")
    try:
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        if result.returncode != 0:
            print("❌ FFmpeg error:")
            print(result.stderr[-2000:])
            return {
                "file": video_file.name,
                "status": "ERROR: ffmpeg failed"
            }
    except Exception as e:
        return {
            "file": video_file.name,
            "status": f"ERROR: {e}"
        }

    # Locate generated segment files
    generated_segments = sorted(temp_folder.glob("segment_*.mp4"))
    copied = 0

    # Rename and copy to local clips output path
    for idx, segment_file in enumerate(generated_segments, start=1):
        if idx > expected_clips:
            # Skip final incomplete segments
            continue

        target_filename = f"{video_stem}_clip_{idx:03d}.mp4"
        destination = output_folder / target_filename

        try:
            shutil.copy2(segment_file, destination)
            copied += 1
        except Exception as e:
            print(f"❌ Copy failed for {target_filename}: {e}")

    # Clean local temporary folder
    try:
        shutil.rmtree(temp_folder, ignore_errors=True)
    except Exception:
        pass

    # Verify results on the final path
    final_count = 0
    for i in range(1, expected_clips + 1):
        clip = output_folder / f"{video_stem}_clip_{i:03d}.mp4"
        if clip.exists() and clip.stat().st_size > 100 * 1024:
            final_count += 1

    if final_count == expected_clips:
        print(f"✅ DONE: {final_count}/{expected_clips} clips saved.")
        return {
            "file": video_file.name,
            "status": "SUCCESS",
            "clips": final_count
        }
    else:
        print(f"⚠️ PARTIAL: {final_count}/{expected_clips} clips saved.")
        return {
            "file": video_file.name,
            "status": "PARTIAL",
            "clips": final_count
        }


# ==============================================================
# 17) FIND ALL VIDEOS
# ==============================================================

all_videos = []

try:
    for type_folder in sorted(INPUT_ROOT.iterdir()):
        if not type_folder.is_dir():
            continue
        for video_file in sorted(type_folder.iterdir()):
            if (
                video_file.is_file()
                and video_file.suffix.lower() in VIDEO_EXTENSIONS
            ):
                all_videos.append(video_file)
except Exception as e:
    print(f"⚠️ Unable to search for videos: {e}")


# ==============================================================
# 18) SHOW INPUT FILES
# ==============================================================

print("\n" + "=" * 72)
print("📦 VIDEOS FOUND")
print("=" * 72)

for i, video in enumerate(all_videos, start=1):
    print(f"{i:03d}. {video.parent.name}/{video.name}")

print(f"\nTotal videos: {len(all_videos)}")
print(f"Parallel workers: {MAX_PARALLEL_VIDEOS}")


# ==============================================================
# 19) PROCESS 10 VIDEOS AT ONCE
# ==============================================================

results = []

if all_videos:
    with ThreadPoolExecutor(max_workers=MAX_PARALLEL_VIDEOS) as executor:
        futures = {
            executor.submit(process_video, video): video
            for video in all_videos
        }
        for future in as_completed(futures):
            video = futures[future]
            try:
                result = future.result()
                results.append(result)
            except Exception as e:
                results.append({
                    "file": video.name,
                    "status": f"ERROR: {e}"
                })
else:
    print("\n⚠️ No videos found.")


# ==============================================================
# 20) FINAL REPORT
# ==============================================================

print("\n")
print("=" * 72)
print("🎉 PROCESSING FINISHED")
print("=" * 72)

success = 0
skipped = 0

errors = 0
partial = 0

for result in results:
    status = result.get("status", "")
    file_name = result.get("file", "")
    clips = result.get("clips", 0)

    if status == "SUCCESS":
        success += 1
        print(f"✅ {file_name} -> {clips} clips")
    elif status.startswith("SKIPPED"):
        skipped += 1
        print(f"⏭️ {file_name} -> {status}")
    elif status == "PARTIAL":
        partial += 1
        print(f"⚠️ {file_name} -> {clips} clips")
    else:
        errors += 1
        print(f"❌ {file_name} -> {status}")


# ==============================================================
# 21) SUMMARY
# ==============================================================

print("\n" + "=" * 72)
print("📊 SUMMARY")
print("=" * 72)

print(f"Total videos : {len(all_videos)}")
print(f"Successful   : {success}")
print(f"Skipped      : {skipped}")
print(f"Partial      : {partial}")

Retrieving folder contents


Retrieving folder 1Wx2p6l-blwnUxHLn_CuJlV1xKR_gBpC5 Bg_Pic
Processing file 1DuckJoTkAxOPtQqyDLjenxPNOLfBiD3v Pvjs.png
Retrieving folder 1xlKVUDUgOasOYNWdOoRC7v25KwfHsQdW logos
Retrieving folder 1v7Q4Oz63QPtdeEjPJd0_WCtFaAaUNRhl funny
Processing file 1a6X8PIbX7B_1NTweDYlphMBlyA2mfq-t 0hfy.png
Retrieving folder 11KarrzBBF-26uoPDpM_TlowkpB70oBOv Islamic
Processing file 1EZwhRDbNqmVmLWaQbEjarDB_qmGkY7cO Bdi.png
Retrieving folder 1q1b44484mbRYXIgo63g6qcglA4oeNQrh Sad
Processing file 1MrGpJ05CMlzski1hYNEl0MMQt7XLnGjI Rid.png
Retrieving folder 1rDHxQdHE9kDhXPne2p8Q6Xj3k76r7WkS outro
Retrieving folder 1bxAg215Ot46xjXL8y2TtuFo-aEpEvx5a funny
Processing file 1w2sWy1srQr_rdJmRLwES9nJvyZoUofgr Copy of Dard_Hota_Hai_Dard_Hone_Do...🖤🥀।_WhatsApp_Status(360p).mp4
Retrieving folder 1UvICvIvCbrgcZ_tZO5__-1Gdju5ZEUJE Islamic
Processing file 1FF-uEuBAX0bxHxnpr4PbOrKBLdwanJVv Copy of Copy of Copy of 20260814_Funny Reaction By Indian Media about Indians Development & Progress😂😅｜｜Funny videos,Try Not to Laug

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1DuckJoTkAxOPtQqyDLjenxPNOLfBiD3v
To: /content/assets/Bg_Pic/Pvjs.png
100%|██████████| 218k/218k [00:00<00:00, 90.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1a6X8PIbX7B_1NTweDYlphMBlyA2mfq-t
To: /content/assets/logos/funny/0hfy.png
100%|██████████| 218k/218k [00:00<00:00, 66.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1EZwhRDbNqmVmLWaQbEjarDB_qmGkY7cO
To: /content/assets/logos/Islamic/Bdi.png
100%|██████████| 21.9k/21.9k [00:00<00:00, 42.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1MrGpJ05CMlzski1hYNEl0MMQt7XLnGjI
To: /content/assets/logos/Sad/Rid.png
100%|██████████| 49.6k/49.6k [00:00<00:00, 39.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1w2sWy1srQr_rdJmRLwES9nJvyZoUofgr
To: /content/assets/outro/funny/Copy of Dard_Hota_Hai_Dard_Hone_Do...🖤🥀।_WhatsApp_Status(360p).mp4
1

Retrieving folder 11TpUkTEK-WgOlAmSz7VI9Usz5WccOxC6 Funny_Cartoon
Processing file 1hnjSKr323Gjp6WQFY_rH31FTwagXHAdK 20260926_Every Indian Street Food Ever !_[7Wi38uVsW98].f616.mp4
Processing file 1HWXZHBdId22XyTc-nLV1n33Q0RSRWjkZ 20260926_Every Indian Street Food Ever !_[7Wi38uVsW98].mp4


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1hnjSKr323Gjp6WQFY_rH31FTwagXHAdK
From (redirected): https://drive.google.com/uc?id=1hnjSKr323Gjp6WQFY_rH31FTwagXHAdK&confirm=t&uuid=e3757e6c-021b-461b-a113-ac3a2e7ac7eb
To: /content/download/Funny_Cartoon/20260926_Every Indian Street Food Ever !_[7Wi38uVsW98].f616.mp4
100%|██████████| 171M/171M [00:00<00:00, 273MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1HWXZHBdId22XyTc-nLV1n33Q0RSRWjkZ
From (redirected): https://drive.google.com/uc?id=1HWXZHBdId22XyTc-nLV1n33Q0RSRWjkZ&confirm=t&uuid=d891233e-85e5-4772-95b3-ddf79248917c
To: /content/download/Funny_Cartoon/20260926_Every Indian Street Food Ever !_[7Wi38uVsW98].mp4
100%|██████████| 180M/180M [00:00<00:00, 220MB/s]
Download completed


Both folders downloaded successfully!
Checking system dependencies...
FFmpeg: ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
All dependencies are ready!
🔄 Installing system dependencies...

🔄 Downloading shared folder contents from Google Drive using gdown...
⚠️ Error during folder download: Failed to retrieve folder contents for folder ID: 1l9ayaXiRBwUr1Hrjag7r3CycVDiN24fh- (status code 404). You may need to change the permission to 'Anyone with the link', or have had many accesses. Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.


Retrieving folder contents



🚀 VIDEO CLIP ENGINE
✅ NVIDIA NVENC GPU detected
⚡ GPU encoding will be used

📦 VIDEOS FOUND
001. Funny_Cartoon/20260926_Every Indian Street Food Ever !_[7Wi38uVsW98].f616.mp4
002. Funny_Cartoon/20260926_Every Indian Street Food Ever !_[7Wi38uVsW98].mp4

Total videos: 2
Parallel workers: 10

------------------------------------------------------------------------
🎬 20260926_Every Indian Street Food Ever !_[7Wi38uVsW98].f616.mp4
📁 Type: Funny_Cartoon

------------------------------------------------------------------------
🎬 20260926_Every Indian Street Food Ever !_[7Wi38uVsW98].mp4
📁 Type: Funny_Cartoon
📐 Resolution: 1920x1080
🎞️ FPS: 30.00
⏱️ Duration: 502.03s
✂️ Full clips: 6
📐 Resolution: 1920x1080
🎞️ FPS: 30.00
⏱️ Duration: 502.06s
✂️ Full clips: 6
⚡ Processing segments...
⚡ Processing segments...
❌ FFmpeg error:

❌ FFmpeg error:



KeyboardInterrupt: 